In [23]:
import ollama
import json
import re
import sys

MODEL_NAME = "llama3.1"


def safe_llm_call(messages, max_tokens=None, temperature=1):
    """Centralized LLM call with error handling."""
    try:
        response = ollama.chat(
            model=MODEL_NAME,
            messages=messages,
            options={
            "temperature": temperature,
            "num_predict": max_tokens
        }
        )
        return response["message"]["content"]
    except Exception as e:
        print(f"\n[LLM ERROR]: {str(e)}")
        return None



In [24]:
# Moderation Check

def regex_moderation_check(text):
    """Fast keyword-based filter."""
    flagged_patterns = [
        r"\bkill\b",
        r"\bpoison\b",
        r"\bweapon\b",
        r"\bdrug?\b",
        r"\bsuicide\b",
        r"\bbomb\b"
    ]

    for pattern in flagged_patterns:
        if re.search(pattern, text, re.IGNORECASE):
            return False
    return True


def llm_moderation_check(text):
    """Context-aware moderation using LLM classification."""

    messages = [
        {
            "role": "system",
            "content": (
                "You are a strict content moderation system. "
                "If the user message contains violence, illegal activity, "
                "self-harm, hate speech, or unsafe instructions, reply only: FLAGGED. "
                "Otherwise reply only: SAFE."
            )
        },
        {"role": "user", "content": text}
    ]

    result = safe_llm_call(messages)

    if result is None:
        return False

    return "SAFE" in result.upper()


def moderate_input(text):
    """Hybrid moderation pipeline."""
    if not regex_moderation_check(text):
        return False

    if not llm_moderation_check(text):
        return False

    return True


In [25]:


def validate_required_field(value, field_name):
    if not value.strip():
        raise ValueError(f"{field_name} cannot be empty.")



def find_possible_recipes(ingredients, cuisine_pref='Any', food_pref='Any', meal_time='Any'):

    messages = [
        {
            "role": "user",
            "content": f"""
            Based only on these ingredients:
            {ingredients}

            Cuisine preference: {cuisine_pref}
            Food preference: {food_pref}
            Meal time: {meal_time}

            Suggest 5 recipe names.
            Return JSON list only.
            """
        }
    ]

    return safe_llm_call(messages)


def health_nutrition_preferences(recipes,
                                 cuisine_pref='Any',
                                 food_pref='Any',
                                 meal_time='Any',
                                 macro_pref=None,
                                 health_conditions=None):

    messages = [
        {
            "role": "user",
            "content": f"""
            Candidate recipes:
            {recipes}

            Cuisine preference: {cuisine_pref}
            Food preference: {food_pref}
            Meal time: {meal_time}
            Macronutrient focus: {macro_pref}
            Health conditions: {health_conditions}

            Select best recipe.
            Return JSON:
            {{
              "selected_recipe": "",
              "reason": ""
            }}
            """
        }
    ]

    return safe_llm_call(messages)



In [38]:

def generate_modified_recipe(selected_recipe,
                             ingredients,
                             cuisine_pref='Any',
                             food_pref='Any',
                             meal_time='Any',
                             macro_pref=None,
                             health_conditions=None,
                             modification_request=None):

    modification_text = f"User modification: {modification_request}" if modification_request else ""

    messages = [
        {
            "role": "user",
            "content": f"""
            Generate a very detailed, beginner-friendly recipe.

            IMPORTANT RULES:


            - Add a summary or overview of the recipe at the top.
            - Give a Traditional but creative name for the recipe.
            - Mention the Portion Size (e.g. 'Serves 2')
            - Do NOT skip any cooking steps.
            - Write steps in atomic detail.
            - Assume the user is an absolute beginner.
            - Include preparation steps like chopping, measuring.
            - Mention exact heat level (low/medium/high).
            - Use more descriptive language for cooking times.
            - Use more description on accompaniments to be used.
            - Mention visual cues (e.g., golden brown, soft texture).
            - Avoid being repititive.
            - Number each step clearly.
            - Give suggestions or modifications of ingredients.
            - Specify the Nutrition value as compared to portion size.
            - Specify various Cooking methods, like using Stove / Pan / Oven, instead of assuming only one.
            - Specify necessary Side dish / Accompaniments wherever needed.

            Recipe: {selected_recipe}
            Ingredients: {ingredients}
            Cuisine: {cuisine_pref}
            Food preference: {food_pref}
            Meal time: {meal_time}
            Macronutrient focus: {macro_pref}
            Health conditions: {health_conditions}

            {modification_text}

            Format as:

            Recipe Name:
            Cuisine:
            Ingredients (with measurements):
            Prep Time:
            Cooking Time:

            Step-by-Step Instructions:

            Nutritional Highlights:
            Substitutions Made:
            """
        }
    ]

    return safe_llm_call(messages)



In [27]:

def eval_recipe_response(recipe_text):

    messages = [
        {
            "role": "user",
            "content": f"""
            Evaluate this recipe:

            {recipe_text}

            Return JSON:
            {{
              "quality_score": "1-10",
              "issues_found": "",
              "improvement_suggestions": ""
            }}
            """
        }
    ]

    return safe_llm_call(messages)



In [ ]:


def collect_user_inputs():

    try:
        ingredients = input("Ingredients (comma separated): ").strip()
        validate_required_field(ingredients, "Ingredients")

        cuisine_pref = input("Cuisine preference (eg: Italian, Asian, Continental, Any) ").strip()
        food_pref = input("Food preference (eg: Veg/Vegan/Non-Veg/Any): ").strip()
        meal_time = input("Meal time (eg: Breakfast/Lunch/Dinner/Snack/Any): ").strip()

        macro_pref = input("Nutrients focus (optional): (eg: High Protein, Low Carb, High Fibre) ").strip()
        health_conditions = input("Health conditions (optional): (eg: Diabetes, PCOS, Heart Friendly) ").strip()

        combined_text = f"{ingredients} {cuisine_pref} {food_pref} {meal_time} {macro_pref} {health_conditions}"

        if not moderate_input(combined_text):
            raise ValueError("Input contains inappropriate or unsafe content.")

        return (ingredients, cuisine_pref, food_pref,
                meal_time, macro_pref, health_conditions)

    except ValueError as ve:
        print(f"\n[INPUT ERROR]: {ve}")
        sys.exit()


def interactive_modification_loop(context):

    while True:
        print("\nPress ENTER to finish without modifications.")
        user_modification = input("Enter modification request: ").strip()

        if user_modification == "":
            print("\nNo further modifications. Exiting.")
            break

        if not moderate_input(user_modification):
            print("\nModification contains unsafe content.")
            continue

        updated_recipe = generate_modified_recipe(
            context["selected_recipe"],
            context["ingredients"],
            context["cuisine_pref"],
            context["food_pref"],
            context["meal_time"],
            context["macro_pref"],
            context["health_conditions"],
            modification_request=user_modification
        )

        print("\n" + "="*60)
        print(updated_recipe)
        print("="*60)

        evaluation = eval_recipe_response(updated_recipe)
        print("\nEvaluation:\n")
        print(evaluation)
        print("="*60)



In [39]:

def main():

    print("\n=== AI Interactive Personalized Recipe System ===\n")


    (ingredients, cuisine_pref, food_pref,
     meal_time, macro_pref, health_conditions) = collect_user_inputs()

    print("\nFinding possible recipes...\n")
    recipes = find_possible_recipes(ingredients, cuisine_pref, food_pref, meal_time)

    print("\nSelecting best recipe...\n")
    selection_json = health_nutrition_preferences(
        recipes,
        cuisine_pref,
        food_pref,
        meal_time,
        macro_pref,
        health_conditions
    )

    try:
        selection_data = json.loads(selection_json)
        selected_recipe = selection_data["selected_recipe"]
    except:
        # print("\n[WARNING] Could not parse selection JSON. Using raw response.")
        selected_recipe = selection_json

    print("\nGenerating recipe...\n")
    final_recipe = generate_modified_recipe(
        selected_recipe,
        ingredients,
        cuisine_pref,
        food_pref,
        meal_time,
        macro_pref,
        health_conditions
    )

    print("\n" + "="*60)
    print(final_recipe)
    print("="*60)

    evaluation = eval_recipe_response(final_recipe)
    print("\nEvaluation:\n")
    print(evaluation)
    print("="*60)

    context = {
        "selected_recipe": selected_recipe,
        "ingredients": ingredients,
        "cuisine_pref": cuisine_pref,
        "food_pref": food_pref,
        "meal_time": meal_time,
        "macro_pref": macro_pref,
        "health_conditions": health_conditions
    }

    interactive_modification_loop(context)

# Run the Program
if __name__ == "__main__":
    main()



=== AI Interactive Personalized Recipe System ===


Finding possible recipes...


Selecting best recipe...


Generating recipe...


**Recipe Name:** North Indian-Style Drumstick Korma
**Cuisine:** North Indian
**Ingredients:**

For 2 servings:

* 4 drumsticks (about 1 pound)
* 2 medium onions, chopped (about 1 cup)
* 3 cloves of garlic, minced (about 2 tablespoons)
* 2 medium tomatoes, diced (about 1 cup)
* 1 teaspoon ground cumin
* 1 teaspoon ground coriander
* 1/2 teaspoon garam masala powder
* 1/2 teaspoon turmeric powder
* 1/4 teaspoon red chili powder
* Salt, to taste
* 2 tablespoons vegetable oil (or ghee)
* 2 tablespoons plain yogurt
* Fresh cilantro, chopped (about 1 tablespoon) for garnish

**Prep Time:** 30 minutes
**Cooking Time:** 25-30 minutes

**Step-by-Step Instructions:**

1. **Prepare the drumsticks**: Rinse the drumsticks under cold water and pat them dry with paper towels. Remove any visible fat or cartilage.
2. **Chop the onions**: Place the chopped onions in a bow